In [ ]:
import numpy as np
from scipy.io import loadmat
import h5py
import matplotlib.pyplot as plt

In [ ]:
raw_path = {
    "train":"Training_dataset_64_64_5000",
    "test":"Testing_dataset_64_64_200",
}
save_path = {
        "train":"trainset.h5",
        "test":"testset.h5",
}

In [ ]:
def minmax_norm(x, xmin, xmax):
    """Min-Max to [-1, 1]"""
    return 2 * (x - xmin) / (xmax - xmin) - 1

In [ ]:
def _obs_field_recons(obs_set, field_size):
    width = obs_set.shape[-1] - 2
    obs_field = np.zeros((width, field_size, field_size))
    for idx in range(obs_set.shape[0]):
        x_pos = int((obs_set[idx,0] + 1) / 2 * (64 - 0) + 0)
        y_pos = int((obs_set[idx,1] + 1) / 2 * (64 - 0) + 0)
        obs_field[:, x_pos, y_pos] = obs_set[idx, 2:]
    return obs_field

### Trainset Processing

In [ ]:
# 1. field generations
data_type = "train"

# target
fun_data = loadmat(f'{raw_path[data_type]}/fun.mat')
kl_terms_data = loadmat(f'{raw_path[data_type]}/kl_terms_all.mat')
phi = fun_data['fun']
xi = kl_terms_data['kl_terms_matrix']

data_len = xi.shape[1]

Lx = 63
Ly = 63
n = Lx + 1
m = Ly + 1

MeanY = 2.0
VarY = 0.5

Y = MeanY + phi @ xi  # (16384×data_len)
K_field = np.exp(Y)     # (16384×data_len)

target = np.zeros((64, 64, data_len))

for i in range(data_len):
    # Transform
    ki = K_field[:, i].reshape(m, n)
    target[:, :, i] = ki

target = np.transpose(target.copy(), (2,0,1))

In [ ]:
with h5py.File(f'{raw_path[data_type]}/C_H_all_full_field_results.h5', "r") as f:
    concentration = f["concentration_data"][:]
    head = f["head_data"][:]

concentration = np.rot90(concentration, k=1, axes=(-2, -1))[..., ::-1, :]
head = np.rot90(head, k=1, axes=(-2, -1))[..., ::-1, :]

### Normalizations

In [ ]:
target_max, target_min = target.max(), target.min()
target_norm_train = minmax_norm(target, target_min, target_max)
concentration_max, concentration_min = concentration.max(), concentration.min()
concentration_norm = minmax_norm(concentration, concentration_min, concentration_max)
head_max, head_min = head.max(), head.min()
head_norm = minmax_norm(head, head_min, head_max)
head_norm = head_norm[:, np.newaxis, :, :]
observation = np.concatenate((concentration_norm, head_norm), axis=1)

### Prepare Sets for Observations

In [ ]:
observation_norm = []

for sample in observation:
    set_list = []
    for x in range(observation.shape[-1]):
        for y in range(observation.shape[-1]):
            element = [minmax_norm(x,0,64),minmax_norm(y,0,64)] + sample[:,x,y].tolist()
            set_list.append(element)
    observation_norm.append(set_list)
observation_norm = np.array(observation_norm)

In [ ]:
with h5py.File(f'../{save_path[data_type]}', "w") as f:
    f.create_dataset("target", data=target_norm_train)
    f.create_dataset("obs", data=observation_norm)
    f.create_dataset("obs_label", data=observation)

### Testset processing

In [ ]:
# 1. field generations
data_type = "test"

# target
fun_data = loadmat(f'{raw_path[data_type]}/fun.mat')
kl_terms_data = loadmat(f'{raw_path[data_type]}/kl_terms_all.mat')
phi = fun_data['fun']
xi = kl_terms_data['kl_terms_matrix']

data_len = xi.shape[1]

Lx = 63
Ly = 63
n = Lx + 1
m = Ly + 1

MeanY = 2.0
VarY = 0.5

Y = MeanY + phi @ xi  # (16384×data_len)
K_field = np.exp(Y)     # (16384×data_len)

target = np.zeros((64, 64, data_len))

for i in range(data_len):
    # Transform
    ki = K_field[:, i].reshape(m, n)
    target[:, :, i] = ki

target = np.transpose(target.copy(), (2,0,1))

In [ ]:
with h5py.File(f'{raw_path[data_type]}/C_H_all_full_field_results.h5', "r") as f:
    concentration = f["concentration_data"][:]
    head = f["head_data"][:]

concentration = np.rot90(concentration, k=1, axes=(-2, -1))[..., ::-1, :]
head = np.rot90(head, k=1, axes=(-2, -1))[..., ::-1, :]

In [ ]:
# target_max, target_min = target.max(), target.min()
target_norm_test = minmax_norm(target, target_min, target_max)
# concentration_max, concentration_min = concentration.max(), concentration.min()
concentration_norm = minmax_norm(concentration, concentration_min, concentration_max)
# head_max, head_min = head.max(), head.min()
head_norm = minmax_norm(head, head_min, head_max)
head_norm = head_norm[:, np.newaxis, :, :]
observation = np.concatenate((concentration_norm, head_norm), axis=1)

In [ ]:
observation_norm = []


for sample in observation:
    set_list = []
    for x in range(observation.shape[-1]):
        for y in range(observation.shape[-1]):
            element = [minmax_norm(x,0,64),minmax_norm(y,0,64)] + sample[:,x,y].tolist()
            set_list.append(element)
    observation_norm.append(set_list)
observation_norm = np.array(observation_norm)

In [ ]:
with h5py.File(f'../{save_path[data_type]}', "w") as f:
    f.create_dataset("target", data=target_norm_test)
    f.create_dataset("obs", data=observation_norm)
    f.create_dataset("obs_label", data=observation)